In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("./WineQT.csv")
print(df.head(5))

x_df = df.drop(columns=["quality", "Id"]).values #удаляем лишнюю колонку
y_df = df["quality"].values
# нормализуем данные
x_sc_df = (x_df - np.min(x_df, axis=0)) / (np.max(x_df, axis=0) - np.min(x_df, axis=0))
print(df.drop(columns=["quality", "Id"]).head(5))

   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0                  54.0   0.9970  3.26       0.65   
3                 17.0                  60.0   0.9980  3.16       0.58   
4                 11.0                  34.0   0.9978  3.51       0.56   

   alcohol  quality  Id  
0      9.4        5   0  
1      9.8        5   1  
2      9

In [2]:
# 80% на учебу, 20% на проверку
X_train, X_test, y_train, y_test = train_test_split(x_sc_df, y_df, test_size=0.2, random_state=42)
y_train = y_train.reshape(-1, 1)

In [3]:
class DenseLayer:
    """Полносвязный слой с обучаемыми весами"""
    def __init__(self, input_dim, output_dim):
        # случайные цифры весов для инициализации
        self.W = np.random.randn(input_dim, output_dim) * 0.1
        # создает матрицу заполненную нулями(1 - это количество строк, а количество столбцов равно количеству нейронов в output_dim), у каждого нейрона появляется своё собственное число-смещение, что бы если Y был бы равен нулю без смещения - всё равно активировался бы нейрон, т.к. смещение добавляет хоть что-то для активации, но ноль на старте, т.к. далее градиентный спуск сам изменит эти нули
        self.b = np.zeros((1, output_dim))
        self.x_sc_df = None
        self.dW = None
        self.db = None

    def forward(self, x_sc_df):
        self.x_sc_df = x_sc_df  # запоминаем вход для backward
        return np.dot(x_sc_df, self.W) + self.b # скалярное произведение X и W + смещение

    def backward(self, output_gradient):
        # x_sc_df.T транспонированная матрица входных данных, output_gradient функция потерь
        self.dW = np.dot(self.x_sc_df.T, output_gradient) # градиент функции потерь по матрице весов dW
        # axis=0 т.к. смещение едино для каждого нейрона), keepdims сохраняет размерность массива
        self.db = np.sum(output_gradient, axis=0, keepdims=True) #суммирование градиентов вдоль нулевой оси
        # передаем градиент ошибки предыдущему слою(следующему в обратном порядке)
        return np.dot(output_gradient, self.W.T)

    def update(self, learning_rate):
        # корректируем в обратную сторону градиенту, т.к. градиент показывает в сторону максимума
        self.W -= learning_rate * self.dW
        self.b -= learning_rate * self.db


class ReLULayer:
    """Слой активации ReLU (отсекает все, что меньше нуля)"""
    def __init__(self):
        self.out = None

    def forward(self, x_sc_df):
        # применяем маску массиву, срезая всё что меньше нуля
        self.out = np.clip(x_sc_df, a_min=0, a_max=None)
        return self.out

    def backward(self, output_gradient):
        # фильтруем, т.к. нельзя использовать то, что меньше нуля
        return output_gradient * (self.out > 0)

# конструктор обучения нейросети
class NeuralNetwork:
    """управляет потоком данных вперед и назад"""
    def __init__(self):
        self.layers = []

    def add(self, layer):
        self.layers.append(layer)

    def forward(self, x_sc_df):
        out = x_sc_df
        for layer in self.layers:
            # применяем активацию слоя, а то что было выходом для одного слоя стало выходом для следующего(для каждого слоя уже задан свой метод forward)
            out = layer.forward(out)
        return out # финальные предсказания последнего слоя

    def backward(self, dL_dy):
        # dL_dy это производная функции ошибки по выходу сети(как сильно изменится расстояние до правильного ответа, если изменим y)
        grad = dL_dy
        for layer in reversed(self.layers):
            # применяем метод backward(который зависит от слоя) каждому слою в обратном порядке
            grad = layer.backward(grad)
        return grad

    def update_weights(self, lr): # lr используется как аргумент при вызове обновления весов
        for layer in self.layers:
            # кладём обновленную скорость обучения в метод update, если у слоя есть метод update
            if hasattr(layer, 'update'):
                layer.update(lr)

In [4]:
# Собираем сеть
model = NeuralNetwork()
# добавляем слои в список слоев, но выходной слой должен иметь такой же размер как входной следующего
model.add(DenseLayer(input_dim=11, output_dim=16))
model.add(ReLULayer()) #размерности нет, просто обрабатывает и отдает столько же, как в прошлом на выходе
model.add(DenseLayer(input_dim=16, output_dim=1))

# параметры обучения
epochs = 800
learning_rate = 0.1

print("Начало обучения ***\n")

for epoch in range(epochs):
    # двигаемся прямо по слоям, применяя функцию forward
    predictions = model.forward(X_train)

    # считаем среднеквадратичную ошибку MSE
    loss = np.mean((predictions - y_train) ** 2)

    # производная функции ошибки MSE по выходу сети
    dL_dy = 2 * (predictions - y_train) / y_train.size

    # двигаемся в обратном направлении(обратное распространение ошибки)
    model.backward(dL_dy)

    # правим веса
    model.update_weights(learning_rate)

    # показываем текущий прогресс каждые 200 полных циклов
    if epoch % 200 == 0:
        print(f"Эпоха {epoch:4d} | Ошибка : {loss:.6f}")

test_predictions = model.forward(X_test)
rounded_predictions = np.round(test_predictions)
accuracy = np.mean(rounded_predictions == y_test.reshape(-1, 1)) * 100
print(f"Итоговая точность модели: {accuracy:.2f}%")

Начало обучения ***

Эпоха    0 | Ошибка : 32.345432
Эпоха  200 | Ошибка : 0.450690
Эпоха  400 | Ошибка : 0.427433
Эпоха  600 | Ошибка : 0.422633
Итоговая точность модели: 62.01%


In [5]:
# Собираем сеть
model = NeuralNetwork()
# добавляем слои в список слоев, но выходной слой должен иметь такой же размер как входной следующего
model.add(DenseLayer(input_dim=11, output_dim=16))
model.add(ReLULayer()) #размерности нет, просто обрабатывает и отдает столько же, как в прошлом на выходе
model.add(DenseLayer(input_dim=16, output_dim=1))

epochs = 5000
learning_rate = 0.1
for epoch in range(epochs):
    # двигаемся прямо по слоям, применяя функцию forward
    predictions = model.forward(X_train)

    # считаем среднеквадратичную ошибку MSE
    loss = np.mean((predictions - y_train) ** 2)

    # производная функции ошибки MSE по выходу сети
    dL_dy = 2 * (predictions - y_train) / y_train.size

    # двигаемся в обратном направлении(обратное распространение ошибки)
    model.backward(dL_dy)

    # правим веса
    model.update_weights(learning_rate)

    # показываем текущий прогресс каждые 200 полных циклов
    if epoch % 200 == 0:
        print(f"Эпоха {epoch:4d} | Ошибка : {loss:.6f}")

test_predictions = model.forward(X_test)
rounded_predictions = np.round(test_predictions)
accuracy = np.mean(rounded_predictions == y_test.reshape(-1, 1)) * 100
print(f"Итоговая точность модели: {accuracy:.2f}%")

Эпоха    0 | Ошибка : 32.873581
Эпоха  200 | Ошибка : 0.432427
Эпоха  400 | Ошибка : 0.467186
Эпоха  600 | Ошибка : 0.418504
Эпоха  800 | Ошибка : 0.413904
Эпоха 1000 | Ошибка : 0.416402
Эпоха 1200 | Ошибка : 0.412752
Эпоха 1400 | Ошибка : 0.411526
Эпоха 1600 | Ошибка : 0.408523
Эпоха 1800 | Ошибка : 0.409165
Эпоха 2000 | Ошибка : 0.406318
Эпоха 2200 | Ошибка : 0.404860
Эпоха 2400 | Ошибка : 0.404238
Эпоха 2600 | Ошибка : 0.403249
Эпоха 2800 | Ошибка : 0.398172
Эпоха 3000 | Ошибка : 0.392732
Эпоха 3200 | Ошибка : 0.393588
Эпоха 3400 | Ошибка : 0.387895
Эпоха 3600 | Ошибка : 0.386405
Эпоха 3800 | Ошибка : 0.397592
Эпоха 4000 | Ошибка : 0.386650
Эпоха 4200 | Ошибка : 0.386577
Эпоха 4400 | Ошибка : 0.388350
Эпоха 4600 | Ошибка : 0.388776
Эпоха 4800 | Ошибка : 0.382996
Итоговая точность модели: 66.38%


При повышении количества эпох - падает точность с понижением ошибки. Это переобучение

In [6]:
# Собираем сеть
model = NeuralNetwork()
# добавляем слои в список слоев, но выходной слой должен иметь такой же размер как входной следующего
model.add(DenseLayer(input_dim=11, output_dim=64))
model.add(ReLULayer()) #размерности нет, просто обрабатывает и отдает столько же, как в прошлом на выходе
model.add(DenseLayer(input_dim=64, output_dim=1))
# параметры обучения
epochs = 1000
learning_rate = 0.1

print("Начало обучения ***\n")

for epoch in range(epochs):
    # двигаемся прямо по слоям, применяя функцию forward
    predictions = model.forward(X_train)

    # считаем среднеквадратичную ошибку MSE
    loss = np.mean((predictions - y_train) ** 2)

    # производная функции ошибки MSE по выходу сети
    dL_dy = 2 * (predictions - y_train) / y_train.size

    # двигаемся в обратном направлении(обратное распространение ошибки)
    model.backward(dL_dy)

    # правим веса
    model.update_weights(learning_rate)

    # показываем текущий прогресс каждые 200 полных циклов
    if epoch % 200 == 0:
        print(f"Эпоха {epoch:4d} | Ошибка : {loss:.6f}")

test_predictions = model.forward(X_test)
rounded_predictions = np.round(test_predictions)
accuracy = np.mean(rounded_predictions == y_test.reshape(-1, 1)) * 100
print(f"Итоговая точность модели: {accuracy:.2f}%")

Начало обучения ***

Эпоха    0 | Ошибка : 33.278052
Эпоха  200 | Ошибка : 0.444254
Эпоха  400 | Ошибка : 0.428660
Эпоха  600 | Ошибка : 0.419027
Эпоха  800 | Ошибка : 0.415155
Итоговая точность модели: 62.01%


Максимальная точность подскочила до 65%(она скачет при перезапусках) при повышении количества слоёв с 16 до 64

In [7]:
# Собираем сеть
model = NeuralNetwork()
# добавляем слои в список слоев, но выходной слой должен иметь такой же размер как входной следующего
model.add(DenseLayer(input_dim=11, output_dim=32))
model.add(ReLULayer()) #размерности нет, просто обрабатывает и отдает столько же, как в прошлом на выходе
model.add(DenseLayer(input_dim=32, output_dim=16))
model.add(ReLULayer())
model.add(DenseLayer(input_dim=16, output_dim=1))
# параметры обучения
epochs = 1000
learning_rate = 0.1

print("Начало обучения ***\n")

for epoch in range(epochs):
    # двигаемся прямо по слоям, применяя функцию forward
    predictions = model.forward(X_train)

    # считаем среднеквадратичную ошибку MSE
    loss = np.mean((predictions - y_train) ** 2)

    # производная функции ошибки MSE по выходу сети
    dL_dy = 2 * (predictions - y_train) / y_train.size

    # двигаемся в обратном направлении(обратное распространение ошибки)
    model.backward(dL_dy)

    # правим веса
    model.update_weights(learning_rate)

    # показываем текущий прогресс каждые 200 полных циклов
    if epoch % 200 == 0:
        print(f"Эпоха {epoch:4d} | Ошибка : {loss:.6f}")

test_predictions = model.forward(X_test)
rounded_predictions = np.round(test_predictions)
accuracy = np.mean(rounded_predictions == y_test.reshape(-1, 1)) * 100
print(f"Итоговая точность модели: {accuracy:.2f}%")

Начало обучения ***

Эпоха    0 | Ошибка : 32.518055
Эпоха  200 | Ошибка : 0.466984
Эпоха  400 | Ошибка : 0.436647
Эпоха  600 | Ошибка : 0.422798
Эпоха  800 | Ошибка : 0.415449
Итоговая точность модели: 58.08%


In [23]:
# Собираем сеть
model = NeuralNetwork()
# добавляем слои в список слоев, но выходной слой должен иметь такой же размер как входной следующего
model.add(DenseLayer(input_dim=11, output_dim=32))
model.add(ReLULayer()) #размерности нет, просто обрабатывает и отдает столько же, как в прошлом на выходе
model.add(DenseLayer(input_dim=32, output_dim=16))
model.add(ReLULayer())
model.add(DenseLayer(input_dim=16, output_dim=1))
# параметры обучения
epochs = 4000
learning_rate = 0.1

print("Начало обучения ***\n")
best_accuracy = 0.0
best_weights = {} # здесь будем хранить копии лучших весов

for epoch in range(epochs):
     # двигаемся прямо по слоям, применяя функцию forward
    predictions = model.forward(X_train)

    # считаем среднеквадратичную ошибку MSE
    loss = np.mean((predictions - y_train) ** 2)

    # производная функции ошибки MSE по выходу сети
    dL_dy = 2 * (predictions - y_train) / y_train.size

    # двигаемся в обратном направлении(обратное распространение ошибки)
    model.backward(dL_dy)

    # правим веса
    model.update_weights(learning_rate)

    # показываем текущий прогресс каждые 200 полных циклов
    if epoch % 500 == 0:
        print(f"Эпоха {epoch:4d} | Ошибка : {loss:.6f}")

    # каждые 100 циклов записываем рекорд
    if epoch % 100 == 0:
        test_preds = model.forward(X_test)
        rounded_preds = np.round(test_preds)
        current_accuracy = np.mean(rounded_preds == y_test.reshape(-1, 1)) * 100

        # если побит рекорд, то обновляем переменные
        if current_accuracy > best_accuracy:
            best_accuracy = current_accuracy
        for idx, layer in enumerate(model.layers):
                if hasattr(layer, 'W'):
                    best_weights[f'layer_{idx}_W'] = layer.W.copy()
                    best_weights[f'layer_{idx}_b'] = layer.b.copy()

for i, layer in enumerate(model.layers):
    if hasattr(layer, 'W'): # проверяем, есть ли у слоя веса (пропускаем ReLU)
        layer.W = best_weights[f'layer_{i}_W']
        layer.b = best_weights[f'layer_{i}_b']

print(f"Итоговая модель восстановлена! Лучшая точность на тесте: {best_accuracy:.2f}%")

Начало обучения ***

Эпоха    0 | Ошибка : 32.587825
Эпоха  500 | Ошибка : 0.430246
Эпоха 1000 | Ошибка : 0.412207
Эпоха 1500 | Ошибка : 0.401944
Эпоха 2000 | Ошибка : 0.396916
Эпоха 2500 | Ошибка : 0.390629
Эпоха 3000 | Ошибка : 0.382692
Эпоха 3500 | Ошибка : 0.376489
Итоговая модель восстановлена! Лучшая точность на тесте: 62.45%


In [9]:
class SupportVectorRegression:
    def __init__(self, learning_rate=0.001, epochs=1000, C=1.0, epsilon=0.1):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.C = C
        self.epsilon = epsilon
        self.W = None
        self.b = None

    def forward(self, X):
        self.X = X  # запоминаем вход для backward
        return np.dot(X, self.W) + self.b

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.W = np.zeros((n_features, 1))
        self.b = 0.0
        y = y.reshape(-1, 1)

        for epoch in range(self.epochs):
            preds = self.forward(X)

            # вычисляем отклонения и маску ошибок
            diff = preds - y
            margin_mask = np.zeros_like(diff)
            margin_mask[diff > self.epsilon] = 1.0
            margin_mask[diff < -self.epsilon] = -1.0

            # усредняем по количеству сэмплов
            dW = (self.W + self.C * np.dot(X.T, margin_mask)) / n_samples
            db = (self.C * np.sum(margin_mask)) / n_samples

            self.W -= self.learning_rate * dW
            self.b -= self.learning_rate * db

    def predict(self, X):
        return np.dot(X, self.W) + self.b

In [24]:
svr_model = SupportVectorRegression(learning_rate=0.01, epochs=2000, C=50.0, epsilon=0.3)

# запускаем обучение
svr_model.fit(X_train, y_train)

# делаем предсказание на тестовых данных
test_preds = svr_model.predict(X_test)

# округляем до целых категорий вин и считаем точность
rounded_preds = np.round(test_preds)
accuracy = np.mean(rounded_preds == y_test.reshape(-1, 1)) * 100

print(f"Точность SVR: {accuracy:.2f}%")

Точность SVR: 61.57%
